# Plant Leaf Disease Detection — Model Training

Trains a CNN on the **PlantVillage** dataset to classify leaf images as healthy or diseased (with the specific disease), using TensorFlow/Keras. Run this notebook in **Google Colab** with a GPU runtime (Runtime → Change runtime type → GPU).

**Output:** `plant_disease_model.h5`, `class_indices.json`, and a starter `disease_info.json` — download all three into your Flask app's `model/` folder.

## 1. Setup

In [ ]:
!pip install -q kaggle tensorflow

import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 2. Get the PlantVillage dataset

Option A (recommended): download via Kaggle. Upload your `kaggle.json` API token when prompted, or mount Drive if you've already downloaded the dataset there.


In [ ]:
from google.colab import files

# Upload your kaggle.json (from kaggle.com/<username>/account -> Create New API Token)
uploaded = files.upload()  # select kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d plantvillage_data

DATASET_DIR = '/content/plantvillage_data'  # adjust if the unzip creates a nested folder
!ls {DATASET_DIR}

> Alternative: `from google.colab import drive; drive.mount('/content/drive')` and point `DATASET_DIR` at a PlantVillage copy already in your Drive.

## 3. Data generators (train/validation split, augmentation)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
)

val_gen = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
)

num_classes = train_gen.num_classes
print("Number of classes:", num_classes)
print(train_gen.class_indices)

## 4. Build the CNN (transfer learning with MobileNetV2)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # freeze for initial training

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## 5. Train

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

EPOCHS = 15

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
)

### Optional: fine-tune
Unfreeze the top layers of MobileNetV2 and train a few more epochs at a low learning rate for extra accuracy.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

fine_tune_history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=6,
    callbacks=callbacks,
)

## 6. Evaluate

In [ ]:
val_loss, val_acc = model.evaluate(val_gen)
print(f"Validation accuracy: {val_acc:.4f}")
print(f"Validation loss: {val_loss:.4f}")

## 7. Export the model, class indices, and a starter disease_info.json

In [ ]:
os.makedirs('export', exist_ok=True)

# 7a. Save the trained model (.h5) for Flask (tf.keras.models.load_model)
model.save('export/plant_disease_model.h5')

# 7b. Save index -> class name mapping (inverse of train_gen.class_indices)
idx_to_class = {str(v): k for k, v in train_gen.class_indices.items()}
with open('export/class_indices.json', 'w') as f:
    json.dump(idx_to_class, f, indent=2)

# 7c. Starter disease_info.json — fill in real descriptions/prevention steps
# for each class label you care about most. Flask falls back to a generic
# message for any label not listed here.
disease_info_template = {
    label: {
        "description": f"TODO: add a description for {label.replace('_', ' ')}.",
        "prevention_steps": [
            "TODO: add prevention/treatment step 1",
            "TODO: add prevention/treatment step 2",
            "TODO: add prevention/treatment step 3",
        ],
    }
    for label in train_gen.class_indices.keys()
}
with open('export/disease_info.json', 'w') as f:
    json.dump(disease_info_template, f, indent=2)

print("Saved to ./export/:")
!ls export

In [ ]:
# Download all three files to your machine, then move them into
# your Flask project's model/ folder.
from google.colab import files as colab_files

colab_files.download('export/plant_disease_model.h5')
colab_files.download('export/class_indices.json')
colab_files.download('export/disease_info.json')